# dim_league.py

## Purpose
Builds `dim_league` as an SCD2 dimension that tracks league name changes across editions.

## Source
- Bronze: `fifa.bronze.ea_fc_players_raw` (column `league`)

## Strategy
- SCD2 with MERGE
- Business key: `league_code`
- Surrogate key: `xxhash64(league_name)`
- Expires the current record when `league_name` changes

## Output
- Table: `fifa.silver.dim_league`

## Columns
- `league_id` (INT)
- `league_code` (STRING)
- `league_name` (STRING)
- `Start_date`, `End_date`, `Is_current` (SCD2 control columns)
- `_ingestion_timestamp` (TIMESTAMP)

## Why SCD2?

Leagues can change names over time (e.g. sponsorship changes: "LaLiga Santander" → "LaLiga EA Sports"). SCD2 keeps the full history of these changes, so you can track which name was valid at each point in time.

## Business key

`league_code` is used as the business key. It stores the original name from EA and is stable across snapshots. If the name changes, the old record is expired and a new one is inserted.

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, lit, regexp_extract, when, concat,trim, explode, split, pmod, xxhash64
from delta.tables import DeltaTable

# ===== Read Bronze =====
bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")

# Extract unique league names
leagues = (
    bronze
    .select(col("league").alias("league_name"))
    .filter(col("league_name").isNotNull() & (col("league_name") != ""))
    .select(trim(col("league_name")).alias("league_name"))
    .distinct()
)

# Build the SCD2-ready DataFrame
new_leagues = (
    leagues
    # Deterministic surrogate key based on the league name
    .withColumn("league_id", pmod(xxhash64(col("league_name")), 1000000).cast("int"))
    .withColumn("league_code", col("league_name"))
    .withColumn("Start_date", current_timestamp())
    .withColumn("End_date", lit(None).cast("timestamp"))
    .withColumn("Is_current", lit(True))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .select(
        "league_id", "league_code", "league_name",
        "Start_date", "End_date", "Is_current", "_ingestion_timestamp"
    )
)

# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.dim_league (
    league_id INT,
    league_code STRING,
    league_name STRING,
    Start_date TIMESTAMP,
    End_date TIMESTAMP,
    Is_current BOOLEAN,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA;
""")

# ===== SCD2 MERGE =====
target = DeltaTable.forName(spark, "fifa.silver.dim_league")

(
    target.alias("t")
    .merge(
        new_leagues.alias("s"),
        "t.league_code = s.league_code AND t.Is_current = true"
    )
    # If the name changed, expire the current record
    .whenMatchedUpdate(
        condition="t.league_name <> s.league_name",
        set={
            "End_date": "s.Start_date",
            "Is_current": "false"
        }
    )
     # Insert new records (or new versions of changed records)
    .whenNotMatchedInsert(
        values={
            "league_id": "s.league_id",
            "league_code": "s.league_code",
            "league_name": "s.league_name",
            "Start_date": "s.Start_date",
            "End_date": "s.End_date",
            "Is_current": "s.Is_current",
            "_ingestion_timestamp": "s._ingestion_timestamp"
        }
    )
    .execute()
)

print(f"dim_league actualizada con {spark.read.table('fifa.silver.dim_league').count()} filas")

LIMIT 10 of the dim_league table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
SELECT * FROM fifa.silver.dim_league ORDER BY league_name;